# Control de cuadratura T96/Q4

Cuaderno diagnóstico reproducible. Estado: preparado y validado estructuralmente, sin ejecutar con los datos del proyecto. La transferencia a Colab requiere autorización específica. La tarea 1 sigue abierta.

## Método y límites

Tres mallas existentes, sin generar h0,28 ni propagar campos. Reglas Gauss–Duffy de 12/16/24 puntos por eje, bloques de 256 triángulos. Controles analíticos de momentos y energías lineales; cotas de discrepancia entre matrices, no de error absoluto frente al continuo. Fuente: contrato incluido en el paquete. Un hilo; versiones fijadas.

In [ ]:
import json, sys, platform, psutil
from datetime import datetime, timezone
print(json.dumps({'utc':datetime.now(timezone.utc).isoformat(),'python':sys.version,'platform':platform.system(),'available_ram_GiB':psutil.virtual_memory().available/2**30},indent=2))

## Preparar las dependencias

Instalación aislada dentro del almacenamiento temporal de Colab. No montar Google Drive ni cambiar recursos de pago. Python observado en Colab: 3.13.16; entorno local: 3.13.7. Se declara esta diferencia y se exige reproducir la matriz histórica antes de adoptar resultados.

In [ ]:
import subprocess
subprocess.run([sys.executable,'-m','pip','install','--target','/content/silice_deps','numpy==2.2.6','scipy==1.15.1','psutil==6.1.1','meshio==5.3.5','scikit-fem==12.0.2'],check=True)

## Cargar el paquete autorizado

Seleccionar únicamente Silice_FEM_Cuadratura_20261008.zip. Verificar su SHA256 y todos los archivos de origen. No cargar credenciales, otros proyectos ni entornos locales.

In [ ]:
from google.colab import files
from pathlib import Path
import hashlib, zipfile
uploaded=files.upload()
assert set(uploaded)=={'Silice_FEM_Cuadratura_20261008.zip'}
archive=Path('Silice_FEM_Cuadratura_20261008.zip')
assert hashlib.sha256(archive.read_bytes()).hexdigest()=='4ae9aa8e48b3176225fb004bee307985220d479c18f29462bacbd41dc427df22'
folder=Path('/content/silice_quadrature_20261008')
assert not folder.exists()
folder.mkdir()
with zipfile.ZipFile(archive) as z:
    assert z.testzip() is None
    assert all(not Path(name).is_absolute() and '..' not in Path(name).parts for name in z.namelist())
    z.extractall(folder)
manifest=json.loads((folder/'transfer_manifest.json').read_text())
project=folder/'project'
for row in manifest['scientific_data']:
    file=project/row['path']
    assert file.stat().st_size==row['bytes'] and hashlib.sha256(file.read_bytes()).hexdigest()==row['sha256']
print({'input_integrity_pass':True,'files_checked':len(manifest['scientific_data'])})

## Ejecutar los tres controles en orden

Las salidas nuevas se conservan. Un fallo detiene el cuaderno; no se sustituye por éxito ni se repite una salida existente. Estos controles no cierran por sí solos la convergencia espacial o longitudinal.

In [ ]:
import os
environment=os.environ.copy()
environment['PYTHONPATH']='/content/silice_deps'
for key in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','NUMEXPR_NUM_THREADS']:
    environment[key]='1'
reports={}
for mesh in ['prototype','coarse','mid']:
    out=project/'resultados/codex'/('cloud_duffy_'+mesh+'_20261008')
    subprocess.run([sys.executable,str(project/'scripts/check_point03_fem_duffy.py'),'--mesh',mesh,'--out',str(out)],env=environment,check=True,timeout=1820)
    reports[mesh]=json.loads((out/'report.json').read_text())

## Revisar los resultados

Los valores son cotas conservadoras de discrepancia entre las matrices calculadas, a 2 mm. La potencia se expresa como fracción de la entrada normalizada. Mantener visibles los fallos y las diferencias de entorno; no afirmar validación experimental.

In [ ]:
rows=[]
for mesh,report in reports.items():
    for pair,values in report['pairs'].items():
        rows.append({'mesh':mesh,'pair':pair,'field_mass_bound':values['field_mass_norm_bound'],'power_field_bound':values['observable_power_bounds']['field'],'power_intensity_bound':values['observable_power_bounds']['intensity'],'pass':values['conservative_consistency_pass']})
print(json.dumps(rows,indent=2))

## Conservar la evidencia

Descargar matrices, informes y fuentes. Verificar los hashes nuevamente al recuperar los resultados localmente y realizar auditoría independiente antes de incorporarlos al estado del proyecto.

In [ ]:
results=Path('/content/Silice_cuadratura_resultados_20261008.zip')
assert not results.exists()
with zipfile.ZipFile(results,'x',zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    z.writestr('environment.json',json.dumps({'python':sys.version,'platform':platform.system(),'utc':datetime.now(timezone.utc).isoformat()},indent=2))
    for file in project.rglob('*'):
        if file.is_file() and 'cloud_duffy_' in str(file):
            z.write(file,file.relative_to(project).as_posix())
print({'result_archive_sha256':hashlib.sha256(results.read_bytes()).hexdigest()})
files.download(str(results))